# Intro

- Recoding of https://github.com/Daniel-EST/deep-steganography/ but updated to match modern tensorflow or pytorch depending what is easiest to use
- from paper: 500 training images, 50 validation images, and 50 test images, all uniformly resized to a manageable dimension of 64x64x3 pixels
    - with rbg colour (no greyscale) to ease training

- overall process:
  - get text which is small enough *after* huffman encoding to fit within 64*64*3 sized block, and encode within image using LSB resulting in **secret image** which will be used by NN (for training all will be encoded with *"Hello World!"* just to keep things simple, but randomized text could be more authentic)
    - LSB - use pillow library (https://pypi.org/project/pillow/) to ease implementation
    - Huffman will also be from GeeksForGeeks / public implementation to speed development
  - NN will be three layers a preperation > hide > reveal networks (with each being NNs) that will ideally allow for full reconstruction of secret image
  - finally undo LSB + Huffman to retrieve final text and check accordingly.

- Issues so far:
  - Deep stenography is greatly outdated so fixing has not been very easy, could promp re-scope of assign to meet deadline if it is unable to be done
  - reformatting between secret image > steg image > revealed secret causes issues with LSB where unless its 100% (>90% ssim) then final classical steganography cannot be performed often
  - could add scheduler to improve flattening learning rate after x epochs and this could benefit the issues with too much noise ruining secret text recovery.

In [4]:
from LSBSteg.LSBSteg import LSBSteg
from dahuffman import HuffmanCodec

# code example from https://github.com/RobinDavid/LSB-Steganography libraru, lsb is using
# LSB 
def LSBEmbed(text,image):
    # Embed text into image using least sig bit method
    steg = LSBSteg(image)
    img_encoded = steg.encode_binary(text)
    return img_encoded

def LSBExtract(image):
    # pull least sig bit from image and reformat into embedded text
    steg = LSBSteg(image)
    extracted_text = steg.decode_binary()
    return extracted_text

# huffman serialization assisted with chatgpt for debugging, dahuffman use from PyPl example
def HuffmanEncode(data):
    # encode text into huffman tree + map
    codec = HuffmanCodec.from_data(data)
    encoded_data = codec.encode(data)

    # finally prepend all so tree:data is one block
    return codec, encoded_data

def HuffmanDecode(codec,data):
    # using Huffman map reconstruct text to original values and decode / unzip data
    decoded_data = codec.decode(data)
    return decoded_data

In [5]:
from datasets import load_dataset
import os, certifi
import numpy as np
import random

os.environ['SSL_CERT_FILE'] = certifi.where()
def is_rgb(example):
    return example['image'].mode == 'RGB'
# option 1 uses "Hello World!" for all to reuse codec and make things simpler
temp_data = load_dataset("zh-plus/tiny-imagenet")
tiny_imageNet_rgb = temp_data.filter(is_rgb)

train_pool = tiny_imageNet_rgb['train']
test_pool = tiny_imageNet_rgb['valid']

ex_data = "Hello World!"
codec,encoded_data = HuffmanEncode(ex_data)

# 2. Sample from Training Pool
num_train = 500
train_indices = random.sample(range(len(train_pool)), num_train * 2)
c_train_imgs = [train_pool[i]['image'] for i in train_indices[:num_train]]
s_train_imgs = [train_pool[i]['image'] for i in train_indices[num_train:]]

# 3. Sample from Test Pool (Audit Set)
num_test = 50
test_indices = random.sample(range(len(test_pool)), num_test * 2)
c_test_imgs = [test_pool[i]['image'] for i in test_indices[:num_test]]
s_test_imgs = [test_pool[i]['image'] for i in test_indices[num_test:]]

s_train_steg = []
for img in s_train_imgs:
    np_img = np.array(img)
    s_train_steg.append( LSBEmbed(encoded_data,np_img))

s_test_steg = []
for img in s_test_imgs:
    np_img = np.array(img)
    s_test_steg.append( LSBEmbed(encoded_data,np_img))

print(f"Pools Ready: {len(c_train_imgs)} Train pairs, {len(c_test_imgs)} Test pairs.")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-1359597a978bc4(…):   0%|          | 0.00/146M [00:00<?, ?B/s]

data/valid-00000-of-00001-70d52db3c749a9(…):   0%|          | 0.00/14.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Pools Ready: 500 Train pairs, 50 Test pairs.


## NN Format etc.
### Input Layer??? (shown in diagram but may not be relevant...)
- IN:
  - secret image: (~,64,64,3)
- OUT:
  - secret image: (~,64,64,3)
### Prep Layer (prep image for embed)
- IN:
  - secret image: (~,64,64,3)
- OUT:
  - prepared secret image (~, 64,64,150)

#### NN Steps
- INIT:
  ```python
    self.conv_layer_4_3x3 = ConvLayer(4, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_layer_4_4x4 = ConvLayer(4, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_layer_4_5x5 = ConvLayer(4, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_1 = tf.keras.layers.Concatenate(axis=3)

    self.conv_1_3x3 = ConvLayer(1, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_1_4x4 = ConvLayer(1, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_1_5x5 = ConvLayer(1, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_2 = tf.keras.layers.Concatenate(axis=3)
  ```
  - what its doing:
    - 2 groups (first 4 channel (batch, h,w,rbg) and second 1 channel (??)) passing to 2d convolution with 50 filters, kernel 3x3 -> 5x5
    - not clear on what 2nd group is for but maybe just more data (so calc between both groups and between 3^2, 4^2, 5^2 matrices allows best quality?)
  
- CALL:
  ```python
    prep_input = tf.keras.layers.Rescaling(1./255, input_shape=input_tensor.shape)(input_tensor)
    conv_4_3x3 = self.conv_layer_4_3x3(prep_input, training=training)
    conv_4_4x4 = self.conv_layer_4_4x4(prep_input, training=training)
    conv_4_5x5 = self.conv_layer_4_5x5(prep_input, training=training)

    concat_1 = self.concat_1([conv_4_3x3, conv_4_4x4, conv_4_5x5])

    conv_1_3x3 =  self.conv_1_3x3(concat_1)
    conv_1_4x4 =  self.conv_1_4x4(concat_1)
    conv_1_5x5 =  self.conv_1_5x5(concat_1)

    return self.concat_2([conv_1_3x3, conv_1_4x4, conv_1_5x5])
  ```
  - what its doing:
    - basically just forward pass for above, so almost same format...
      - scaling 1./255 is interesting but not clear on why but google says normalizes pixels to log 0-1 vs 0-255 so making sure these are kept as tensors I guess.
### Hide Layer (embed prepared secret with cover)
- IN:
  - prepared secret image (~, 64,64,150)
  - cover image (~, 64,64,3)
- OUT:
  - hidden/steg image: (~64,64,3)

#### NN Steps
- IN:
  ```python
    self.prep_layer = PrepLayer()
    self.concat_1 = tf.keras.layers.Concatenate(axis=3)

    self.conv_layer_4_3x3 = ConvLayer(4, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_layer_4_4x4 = ConvLayer(4, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_layer_4_5x5 = ConvLayer(4, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_2 = tf.keras.layers.Concatenate(axis=3)

    self.conv_1_3x3 = ConvLayer(1, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_1_4x4 = ConvLayer(1, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_1_5x5 = ConvLayer(1, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_3 = tf.keras.layers.Concatenate(axis=3)

    self.conv_1_1x1 = ConvLayer(1, filters=3, kernel_size=(1, 1), activation=tf.nn.relu)
  ```
  - what its doing:
    - similar to prep but with final conv after, and calling prep then concat before NN steps?
- CALL:
  ```python
    prep_input = input_tensor[0]
    hide_input = tf.keras.layers.Rescaling(1./255, input_shape=input_tensor[1].shape)(input_tensor[1])
    concat_1 = self.concat_1([prep_input, hide_input])

    conv_4_3x3 = self.conv_layer_4_3x3(concat_1, training=training)
    conv_4_4x4 = self.conv_layer_4_4x4(concat_1, training=training)
    conv_4_5x5 = self.conv_layer_4_5x5(concat_1, training=training)

    concat_2 = self.concat_2([conv_4_3x3, conv_4_4x4, conv_4_5x5])

    conv_1_3x3 =  self.conv_1_3x3(concat_2)
    conv_1_4x4 =  self.conv_1_4x4(concat_2)
    conv_1_5x5 =  self.conv_1_5x5(concat_2)

    concat_3 = self.concat_3([conv_1_3x3, conv_1_4x4, conv_1_5x5])

    return self.conv_1_1x1(concat_3)
  ```
  - what its doing:
    - just forward for hide like with prep...
### Reveal Layer (decode / recover secret)
- IN:
  - hidden/steg image: (~,64,64,3)
- OUT:
  - recovered secret image: (~,64,64,3)
#### NN Steps
- IN:
  ```python
    self.conv_layer_4_3x3 = ConvLayer(4, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_layer_4_4x4 = ConvLayer(4, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_layer_4_5x5 = ConvLayer(4, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_1 = tf.keras.layers.Concatenate(axis=3)

    self.conv_1_3x3 = ConvLayer(1, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_1_4x4 = ConvLayer(1, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_1_5x5 = ConvLayer(1, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_2 = tf.keras.layers.Concatenate(axis=3)

    self.conv_1_1x1 = ConvLayer(1, filters=3, kernel_size=(1, 1), activation=tf.nn.relu)
  ```
  - what its doing:
    - same as prep but using hidden image so this is training to undo... but otherwise the same thing basically
- CALL:
  ```python
    conv_4_3x3 = self.conv_layer_4_3x3(input_tensor, training=training)
    conv_4_4x4 = self.conv_layer_4_4x4(input_tensor, training=training)
    conv_4_5x5 = self.conv_layer_4_5x5(input_tensor, training=training)

    concat_1 = self.concat_1([conv_4_3x3, conv_4_4x4, conv_4_5x5])

    conv_1_3x3 =  self.conv_1_3x3(concat_1)
    conv_1_4x4 =  self.conv_1_4x4(concat_1)
    conv_1_5x5 =  self.conv_1_5x5(concat_1)

    concat_2 = self.concat_2([conv_1_3x3, conv_1_4x4, conv_1_5x5])

    return self.conv_1_1x1(concat_2)
  ```
  - what its doing:
    - forward of above

### Steg Loss
- IN:
  - secret and reveal MSE values / model to derive MSEs
- OUT:
  - loss value in MSE for training
#### Code
- IN:
  ```python
      beta = tf.constant(self.beta, name='beta')

      secret_true = y_true[0]
      secret_pred = y_pred[0]

      cover_true = y_true[1]
      cover_pred = y_pred[1]

      secret_mse = tf.losses.MSE(secret_true, secret_pred)
      cover_mse = tf.losses.MSE(cover_true, cover_pred)

      return tf.reduce_mean(cover_mse + beta * secret_mse)
  ```
  - what its doing:
    - take beta (as constant) and then do cover + secret with reduce mean func to find final MSE, balanced between both. kinda just average mse between secret and reveal not too complex

### Training
- epochs: 100
- Batch size = 32
- LR: 0.001
- adam + steg loss
- suffling buffer size: 64 (maybe shuffle=true in code)

### Testing
- just run model.predict on test gen

In [29]:
import tensorflow as tf
from tensorflow.keras import layers, losses

class SteganoModel(tf.keras.Model):
    def __init__(self):
        super(SteganoModel, self).__init__()
        self.beta = 1
        
        # 1. PREP NETWORK: (None, 64, 64, 3) -> (None, 64, 64, 150)
        # Using 50 filters of each size (3x3, 4x4, 5x5) to reach 150
        self.prep_conv3 = layers.Conv2D(50, (3, 3), padding='same', activation='relu')
        self.prep_conv4 = layers.Conv2D(50, (4, 4), padding='same', activation='relu')
        self.prep_conv5 = layers.Conv2D(50, (5, 5), padding='same', activation='relu')

        # 2. HIDE NETWORK: (None, 64, 64, 150+3) -> (None, 64, 64, 3)
        self.hide_conv1 = layers.Conv2D(50, (3, 3), padding='same', activation='relu')
        self.hide_conv2 = layers.Conv2D(10, (3, 3), padding='same', activation='relu')
        self.hide_conv3 = layers.Conv2D(5, (3, 3), padding='same', activation='relu')
        self.hide_final = layers.Conv2D(3, (3, 3), padding='same', activation='sigmoid')

        # 3. REVEAL NETWORK: (None, 64, 64, 3) -> (None, 64, 64, 3)
        self.reveal_conv1 = layers.Conv2D(50, (3, 3), padding='same', activation='relu')
        self.reveal_conv2 = layers.Conv2D(10, (3, 3), padding='same', activation='relu')
        self.reveal_conv3 = layers.Conv2D(5, (3, 3), padding='same', activation='relu')
        self.reveal_final = layers.Conv2D(3, (3, 3), padding='same', activation='sigmoid')

    def hide(self, secret, cover):
        # Prep
        p3 = self.prep_conv3(secret)
        p4 = self.prep_conv4(secret)
        p5 = self.prep_conv5(secret)
        prep_out = tf.concat([p3, p4, p5], axis=-1) # 150 channels
        
        # Hide
        merged = tf.concat([prep_out, cover], axis=-1) # 153 channels
        x = self.hide_conv1(merged)
        x = self.hide_conv2(x)
        x = self.hide_conv3(x)
        return self.hide_final(x)

    def reveal(self, stego):
        x = self.reveal_conv1(stego)
        x = self.reveal_conv2(x)
        x = self.reveal_conv3(x)
        return self.reveal_final(x)

    def call(self, inputs):
        cover, secret = inputs
        stego = self.hide(secret, cover)
        extracted = self.reveal(stego)
        return stego, extracted

    def train_step(self, data):
        # Unpack the data (Expecting cover and secret images)
        cover, secret = data

        with tf.GradientTape() as tape:
            stego, extracted = self((cover, secret), training=True)
            
            # Loss Calculation: total_loss = cover_mse + beta * secret_mse
            cover_mse = tf.reduce_mean(losses.mse(cover, stego))
            secret_mse = tf.reduce_mean(losses.mse(secret, extracted))
            total_loss = cover_mse + (self.beta * secret_mse)

        # Apply gradients
        trainable_vars = self.trainable_variables
        gradients = tape.gradient(total_loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))
        
        return {"loss": total_loss, "cover_mse": cover_mse, "secret_mse": secret_mse}

# Initialize
model = SteganoModel()

In [24]:
# --- Training Configuration ---
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE))

x_train_cover = np.array([np.array(img) for img in c_train_imgs]).astype('float32') / 255.0
x_train_secret = np.array([np.array(img) for img in s_train_steg]).astype('float32') / 255.0

train_ds = tf.data.Dataset.from_tensor_slices((x_train_cover, x_train_secret))
train_ds = train_ds.shuffle(buffer_size=64).batch(BATCH_SIZE)

history = model.fit(
    train_ds, 
    epochs=EPOCHS,
    steps_per_epoch=BATCH_SIZE 
)

print("Training Complete.")

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 12s 296ms/step - cover_mse: 0.0212 - loss: 0.0406 - secret_mse: 0.0194
Epoch 2/10


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 293ms/step - cover_mse: 0.0185 - loss: 0.0405 - secret_mse: 0.0220
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 292ms/step - cover_mse: 0.0160 - loss: 0.0316 - secret_mse: 0.0156
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 11s 339ms/step - cover_mse: 0.0217 - loss: 0.0358 - secret_mse: 0.0141
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 400ms/step - cover_mse: 0.0204 - loss: 0.0380 - secret_mse: 0.0177
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 396ms/step - cover_mse: 0.0187 - loss: 0.0377 - secret_mse: 0.0190
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 400ms/step - cover_mse: 0.0202 - loss: 0.0363 - secret_mse: 0.0162
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 14s 410ms/step - cover_mse: 0.0184 - loss: 0.0312 - secret_mse: 0.0128
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 14s 409ms/step - cover_mse: 0.0201 - loss: 0.0368 - secret_mse: 0.0167
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 392ms/step - cover_mse: 0.0142 - loss: 0.0288 - secret_mse: 0.0146
Training Complete.


In [28]:
# Save the whole model weights
model.save('steganography_model.keras')

# so that reveal is seperate?
# 1. Define a fresh input for the standalone decoder
decoder_input = tf.keras.Input(shape=(64, 64, 3), name='stego_input_only')

# 2. Extract the reveal logic from your trained model
# We pass the new input through the reveal_layer of the existing 'model'
decoder_output = model.reveal(decoder_input)

# 3. Create the separate model
reveal_model = tf.keras.models.Model(inputs=decoder_input, outputs=decoder_output)

# 4. Compile and Save
reveal_model.compile(optimizer='adam', loss='mse')
reveal_model.save('standalone_reveal.keras')